# Masked attention on persistence diagrams

This notebook uses the corrected preprocessing and the same 180/60/60 partitions as the logistic regression baselines. The historical experiments remain in `attention_TDA.ipynb`. Run from the project root with the `.venv` kernel.


## Architecture

Birth/death points → 32-dimensional projection + learned CLS → two masked attention encoders → CLS classifier. No positional encoding is used: diagram points form an unordered set. Each encoder masks padded keys and resets padded query states.


In [ ]:
from attention_model import build_attention_model

model = build_attention_model()
model.summary()


## Training protocol

Scaling is fitted on training points only, before padding. Adam uses a learning rate of 0.001 and batches of 16. Early stopping selects the minimum validation loss (patience 10, at most 50 epochs). The selected weights are restored before the test is evaluated. ATOL is not used here.


In [ ]:
from train_attention import run_attention

# Train using the same split as the baselines and save the selected checkpoint.
report = run_attention(seed=42, epochs=50, batch_size=16)


In [ ]:
from pathlib import Path
from IPython.display import Markdown, display

display(Markdown(Path("results/attention/comparison.md").read_text()))


## Learning curves

The curves show the full training history. The reported scores above use the best validation-loss checkpoint, not necessarily the last epoch.


In [ ]:
import matplotlib.pyplot as plt

history = report["history"]
epochs = range(1, report["epochs_run"] + 1)
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, metric in zip(axes, ["loss", "accuracy"]):
    ax.plot(epochs, history[metric], label="train")
    ax.plot(epochs, history["val_" + metric], label="validation")
    ax.axvline(report["best_epoch"], color="gray", linestyle="--", label="selected epoch")
    ax.set(xlabel="Epoch", ylabel=metric)
    ax.legend()
plt.tight_layout()
